In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/PoultryVision"
os.makedirs(BASE_FOLDER, exist_ok=True)

print(f"Folder creado: {BASE_FOLDER}")

Folder creado: /content/drive/MyDrive/PoultryVision


In [ ]:
import torch

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

GPU: NVIDIA A100-SXM4-40GB
VRAM (GB): 42.405855232


In [ ]:
import os
import zipfile
from pathlib import Path

In [ ]:
!nvidia-smi

Fri Jun  5 03:15:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             41W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Directorios del proyecto
BASE_FOLDER   = Path("/content/drive/MyDrive/PoultryVision")
RAW_DIR        = BASE_FOLDER / "datasets" / "PIO" / "raw"
EXTRACT_DIR    = "./pio_dataset"


In [ ]:
import subprocess
from pathlib import Path

In [ ]:
def extract_rar(rar_path: Path, dest_dir: Path) -> bool:
    """Extract .rar to dir"""
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n Extracting : {rar_path.name} \u2192 {dest_dir}")

    # Intentar con unrar, fallback a 7z
    for cmd in [
        ["unrar", "x", "-y", str(rar_path), str(dest_dir) + "/"],
        ["7z",    "x", str(rar_path), f"-o{dest_dir}", "-y"],
    ]:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f" Completed")
            return True
        print(f"{cmd[0]} Failed..")

    print(f"Unabled to extract files : {rar_path.name}")
    print(result.stderr[-500:] if result.stderr else "")
    return False

In [ ]:
extract_results = {}


In [ ]:

# data.rar → extracted/data/
data_rar = RAW_DIR / "data.rar"
if data_rar.exists():
    ok = extract_rar(data_rar, Path(EXTRACT_DIR) )
    extract_results["data.rar"] = ok
else:
    print("  data.rar not found ")
    extract_results["data.rar"] = False

# Test 2025.rar → extracted/test_2025/
test_rar = RAW_DIR / "Test 2025.rar"
if test_rar.exists():
    ok = extract_rar(test_rar, Path(EXTRACT_DIR) / "test_2025")
    extract_results["Test 2025.rar"] = ok
else:
    print("   Test 2025.rar not found , skipping")
    extract_results["Test 2025.rar"] = False

print("Summary:")
for name, ok in extract_results.items():
    print(f"   {'ok' if ok else 'fail'} {name}")


 Extracting : data.rar → pio_dataset
 Completed

 Extracting : Test 2025.rar → pio_dataset/test_2025
 Completed
Summary:
   ok data.rar
   ok Test 2025.rar


In [ ]:
from collections import defaultdict, Counter

In [ ]:
def inspect_directory(root: Path, max_depth: int = 4) -> dict:
    """DEEP DIR INSPECTION"""
    stats = {
        "total_files":  0,
        "by_extension": Counter(),
        "by_folder":    defaultdict(int),
        "image_count":  0,
        "label_count":  0,
        "total_size_mb": 0.0,
    }
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    label_exts = {".txt"}

    for path in root.rglob("*"):
        if not path.is_file():
            continue
        ext  = path.suffix.lower()
        size = path.stat().st_size
        rel  = path.relative_to(root)
        depth = len(rel.parts)

        if depth > max_depth:
            continue

        stats["total_files"]   += 1
        stats["by_extension"][ext] += 1
        stats["by_folder"][str(rel.parent)] += 1
        stats["total_size_mb"] += size / (1024**2)

        if ext in image_exts:
            stats["image_count"] += 1
        if ext in label_exts:
            stats["label_count"] += 1

    return stats

In [ ]:
print("\n" + "="*60)
print("CHECKING DATASET STRUCTURE")
print("="*60)
print("\nExtracted/data/:")
data_dir = Path(EXTRACT_DIR)
if data_dir.exists():
    stats = inspect_directory(data_dir)
    print(f"   Total Files:  {stats['total_files']:,}")
    print(f"   Images        {stats['image_count']:,}")
    print(f"   Labels (.txt):   {stats['label_count']:,}")
    print(f"   Size :    {stats['total_size_mb']:.1f} MB")
    print(f"\n  Ext :")
    for ext, count in stats["by_extension"].most_common():
        print(f"     {ext or '(sin ext)':12s}  {count:6,} files")
    print(f"\nFolders :")
    for folder, count in sorted(stats["by_folder"].items()):
        print(f"     {folder:40s}  {count:6,} files ")
else:
    print("Dir was not found , Retry again")


CHECKING DATASET STRUCTURE

Extracted/data/:
   Total Files:  3,052
   Images        1,560
   Labels (.txt):   1,489
   Size :    1537.0 MB

  Ext :
     .jpg           1,560 files
     .txt           1,489 files
     .cache             2 files
     .yaml              1 files

Folders :
     data                                           2 files 
     data/images/train                          1,035 files 
     data/images/val                              452 files 
     data/labels                                    2 files 
     data/labels/train                          1,036 files 
     data/labels/val                              452 files 
     test_2025/Test 2025                           73 files 


In [ ]:
# Inspeccionar extracted/test_2025
print("\n Extracted/test_2025/:")
test_dir = Path(EXTRACT_DIR) / "test_2025"
if test_dir.exists():
    stats_test = inspect_directory(test_dir)
    print(f"   Total files :  {stats_test['total_files']:,}")
    print(f"   Imagess:        {stats_test['image_count']:,}")
    print(f"   Labels (.txt):   {stats_test['label_count']:,}")
    print(f"   Size:    {stats_test['total_size_mb']:.1f} MB")
else:
    print(" Dir was not found ")


 Extracted/test_2025/:
   Total files :  73
   Imagess:        73
   Labels (.txt):   0
   Size:    67.6 MB


In [ ]:
def inspect_yolo_labels(labels_dir: Path, n_samples: int = 10) -> dict:
    """
    Check .txt files to YOLO format .
    Expected: class_id cx cy w h
    """
    stats = {
        "files_checked": 0,
        "total_annotations": 0,
        "classes": Counter(),
        "empty_files": 0,
        "malformed": 0,
        "bbox_w_mean": [],
        "bbox_h_mean": [],
    }

    txt_files = list(labels_dir.rglob("*.txt"))
    if not txt_files:
        print("Not found .txt")
        return stats

    sample = txt_files[:n_samples]
    print(f"   .txt found: {len(txt_files):,}")
    print(f"   show sample {len(sample)} files s:\n")

    for txt_path in sample:
        lines = txt_path.read_text().strip().split("\n")
        lines = [l for l in lines if l.strip()]

        if not lines:
            stats["empty_files"] += 1
            continue

        stats["files_checked"] += 1
        print(f"{txt_path.name}  ({len(lines)} anotaciones)")

        for line in lines[:3]:  # máx 3 anotaciones por archivo
            parts = line.strip().split()
            if len(parts) != 5:
                stats["malformed"] += 1
                print(f" not valid: {line}")
                continue
            cls, cx, cy, w, h = parts
            stats["total_annotations"] += 1
            stats["classes"][int(cls)] += 1
            stats["bbox_w_mean"].append(float(w))
            stats["bbox_h_mean"].append(float(h))
            print(f"      clase={cls}  cx={float(cx):.4f}  cy={float(cy):.4f}  "
                  f"w={float(w):.4f}  h={float(h):.4f}")
        if len(lines) > 3:
            print(f"      ... && {len(lines)-3} extra annotation")

    # Stats globales (todos los archivos)
    print(f"\n  Overall Stats :")
    all_annotations = 0
    all_classes = Counter()
    all_w, all_h = [], []

    for txt_path in txt_files:
        lines = txt_path.read_text().strip().split("\n")
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, cx, cy, w, h = parts
                all_annotations += 1
                all_classes[int(cls)] += 1
                all_w.append(float(w))
                all_h.append(float(h))

    import numpy as np
    print(f"   Total Annotations: {all_annotations:,}")
    print(f"   Classes:  {dict(all_classes)}")
    print(f"   BBox avg Width:  {np.mean(all_w):.4f} ± {np.std(all_w):.4f}")
    print(f"   BBox AVG Height:   {np.mean(all_h):.4f} ± {np.std(all_h):.4f}")
    print(f"   BBox AVG Area :   {np.mean([w*h for w,h in zip(all_w, all_h)]):.6f}")
    print(f"   ( Tiny Areas = dense crowded chickens top-down)")

    return {
        "total_annotations": all_annotations,
        "classes": dict(all_classes),
        "n_label_files": len(txt_files),
    }

In [ ]:
# Buscar directorio de labels en extracted
labels_dir = None
for candidate in [
    Path(EXTRACT_DIR) / "data" / "labels",
    Path(EXTRACT_DIR) / "data" / "labels" / "train",
    Path(EXTRACT_DIR) / "data",
]:
    if candidate.exists() and list(candidate.rglob("*.txt")):
        labels_dir = candidate
        break

if labels_dir:
    print(f" Found labels: {labels_dir}")
    label_stats = inspect_yolo_labels(labels_dir)
else:
    print("No Found Dirs \n Check Again.")
    label_stats = {}

 Found labels: pio_dataset/data/labels
   .txt found: 1,488
   show sample 10 files s:

C-W1-0009.txt  (425 anotaciones)
      clase=0  cx=0.4547  cy=0.7810  w=0.0198  h=0.0528
      clase=0  cx=0.4154  cy=0.8403  w=0.0297  h=0.0398
      clase=0  cx=0.6656  cy=0.8245  w=0.0219  h=0.0306
      ... && 422 extra annotation
C-W1-0132.txt  (479 anotaciones)
      clase=0  cx=0.4893  cy=0.4375  w=0.0214  h=0.0565
      clase=0  cx=0.4010  cy=0.3935  w=0.0219  h=0.0500
      clase=0  cx=0.5898  cy=0.5134  w=0.0214  h=0.0435
      ... && 476 extra annotation
C-W2-0078.txt  (248 anotaciones)
      clase=0  cx=0.6016  cy=0.0843  w=0.0344  h=0.0444
      clase=0  cx=0.7768  cy=0.1347  w=0.0266  h=0.0602
      clase=0  cx=0.4909  cy=0.1028  w=0.0286  h=0.0630
      ... && 245 extra annotation
C-W2-0190.txt  (297 anotaciones)
      clase=0  cx=0.6393  cy=0.2310  w=0.0359  h=0.0694
      clase=0  cx=0.2010  cy=0.8968  w=0.0302  h=0.0528
      clase=0  cx=0.3560  cy=0.2319  w=0.0359  h=0.0454
      

In [ ]:
import shutil
from pathlib import Path
from tqdm import tqdm

for split in ["train", "val"]:
    # Correct source paths to include the 'data' subdirectory
    src_imgs = Path(EXTRACT_DIR) / "data" / "images" / split
    src_lbls = Path(EXTRACT_DIR) / "data" / "labels" / split
    dst_imgs = Path("dataset") / "images" / split
    dst_lbls = Path("dataset") / "labels" / split
    dst_imgs.mkdir(parents=True, exist_ok=True)
    dst_lbls.mkdir(parents=True, exist_ok=True)

    imgs = sorted(src_imgs.glob("*.jpg")) + sorted(src_imgs.glob("*.png"))
    print(f"{split}: {len(imgs)} images")

    for img in tqdm(imgs, desc=split):
        lbl = src_lbls / (img.stem + ".txt")
        shutil.copy2(img, dst_imgs / img.name)
        if lbl.exists():
            shutil.copy2(lbl, dst_lbls / lbl.name)

# Test set comes from Test 2025.rar — only images, no labels
# Correct source path for test images
src_test_imgs_dir = Path(EXTRACT_DIR) / "test_2025" / "Test 2025"
dst_test = Path("dataset") / "images" / "test"
dst_test.mkdir(parents=True, exist_ok=True)

test_imgs = sorted(src_test_imgs_dir.glob("*.jpg")) + sorted(src_test_imgs_dir.glob("*.png"))
print(f"test: {len(test_imgs)} images (no labels)")
for img in tqdm(test_imgs, desc="test"):
    shutil.copy2(img, dst_test / img.name)

# Verificar resultado
print("\nFinal YOLO structure:")
for split in ["train", "val", "test"]:
    n_img = len(list((Path("dataset") / "images" / split).glob("*")))
    n_lbl = len(list((Path("dataset") / "labels" / split).glob("*.txt")))
    print(f"  {split:6s}:  {n_img:5} images  |  {n_lbl:5} labels")

train: 1035 images


train: 100%|██████████| 1035/1035 [00:00<00:00, 1314.32it/s]


val: 452 images


val: 100%|██████████| 452/452 [00:00<00:00, 672.88it/s]


test: 73 images (no labels)


test: 100%|██████████| 73/73 [00:00<00:00, 635.27it/s]


Final YOLO structure:
  train :   1035 images  |   1035 labels
  val   :    452 images  |    452 labels
  test  :     73 images  |      0 labels


Now that the dataset is reorganized into the standard YOLO structure, we can clean up the intermediate extracted directories.

In [ ]:
import shutil
from pathlib import Path

In [ ]:

# Define the paths to the old directories
old_data_dir = Path(EXTRACT_DIR) / "data"
old_test_2025_dir = Path(EXTRACT_DIR) / "test_2025"

# Remove the directories if they exist
if old_data_dir.exists():
    print(f"Deleting old directory: {old_data_dir}")
    shutil.rmtree(old_data_dir)
else:
    print(f"Old data directory not found: {old_data_dir}")

if old_test_2025_dir.exists():
    print(f"Deleting old directory: {old_test_2025_dir}")
    shutil.rmtree(old_test_2025_dir)
else:
    print(f"Old test_2025 directory not found: {old_test_2025_dir}")

print("Cleanup complete.")

Deleting old directory: pio_dataset/data
Deleting old directory: pio_dataset/test_2025
Cleanup complete.


In [ ]:
from datetime import datetime

In [ ]:
yaml_content = f"""# PoultryVision — PIO Dataset
# Generado automáticamente por ETL_stage0.py
# Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
#
# Dataset: PIO - Poultry Images for Object detection
# Paper: https://www.nature.com/articles/s41597-026-07114-5
# Zenodo: https://doi.org/10.5281/zenodo.16686320

path: { EXTRACT_DIR}

train: images/train
val:   images/val
test:  images/test

# Número de clases
nc: 1

# Nombres de clases
names:
  0: chicken

# dataser come from PIO Broiler Chicken dataset
"""


In [ ]:
from pathlib import Path

yaml_path = Path(EXTRACT_DIR) / "dataset.yaml"
yaml_path.write_text(yaml_content)

464

In [ ]:
print(f"dataset.yaml generado: {yaml_path}")
print()
print(yaml_content)


dataset.yaml generado: pio_dataset/dataset.yaml

# PoultryVision — PIO Dataset
# Generado automáticamente por ETL_stage0.py
# Fecha: 2026-06-04 20:51:18
#
# Dataset: PIO - Poultry Images for Object detection
# Paper: https://www.nature.com/articles/s41597-026-07114-5
# Zenodo: https://doi.org/10.5281/zenodo.16686320

path: ./pio_dataset

train: images/train
val:   images/val
test:  images/test

# Número de clases
nc: 1

# Nombres de clases
names:
  0: chicken

# dataser come from PIO Broiler Chicken dataset



In [ ]:
from pathlib import Path
import os

def count_split(split_name):
    imgs = list((Path(EXTRACT_DIR) / "images" / split_name).glob("*"))
    lbls = list((Path(EXTRACT_DIR) / "labels" / split_name).glob("*.txt"))
    return len(imgs), len(lbls)

In [ ]:
print(f"\n Estructura YOLO-ready dataset :")
for split in ["train", "val", "test"]:
    n_img, n_lbl = count_split(split)
    print(f"   {split:6s}: {n_img:5,} imágenes  |  {n_lbl:5,} labels")


 Estructura YOLO-ready dataset :
   train : 1,035 imágenes  |  1,035 labels
   val   :   452 imágenes  |    452 labels
   test  :    73 imágenes  |      0 labels


In [ ]:
import yaml

In [ ]:
yaml_path = f"{EXTRACT_DIR}/dataset.yaml"

In [ ]:
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
config['path'] = EXTRACT_DIR
config['train'] = 'images/train'
config['val'] = 'images/val'
config['test'] = 'images/test'

In [ ]:
with open(yaml_path, 'w') as f:
    yaml.safe_dump(config, f)

print("PIO DATASET UPDATE.")

In [ ]:
yaml_path = f"{EXTRACT_DIR}/dataset.yaml"
print(yaml_path)

./pio_dataset/dataset.yaml


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
v1_weights_path = "/content/drive/MyDrive/PoultryVision/chicken_v1/weights/best.pt"

In [ ]:
model = YOLO(v1_weights_path)

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=60,
    imgsz=1280,
    batch=8,
    rect=True,
    cache=True,
    workers=8,
    optimizer='AdamW',
    lr0=0.0016,
    lrf=0.01,
    box=7,
    cls=1.5,
    mosaic=0,
    patience=10,

    verbose=True,
    project=BASE_FOLDER,
    name="chicken_PIO_V3",
    exist_ok=True
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7, cache=True, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./pio_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0016, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/PoultryVision/chicken_v1/weights/best.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=chicken_PIO_V3, nbs=64, nms=False, opset=No

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.9it/s 7.5s
                   all        452      73859      0.949      0.869      0.894      0.666

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/60      38.5G     0.9071      1.445     0.9393          3       1280: 100% ━━━━━━━━━━━━ 130/130 5.5it/s 23.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.2it/s 4.7s
                   all        452      73859      0.952      0.862      0.894      0.666

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/60      37.2G     0.9015      1.431     0.9392          3       1280: 100% ━━━━━━━━━━━━ 130/130 5.5it/s 23.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.1it/s 4.7s
                   all        452      73859 

In [ ]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4149.6±900.5 MB/s, size: 1150.0 KB)
val: Scanning /content/pio_dataset/labels/val.cache... 452 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 452/452 158.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 10% ━─────────── 3/29 2.3s/it 5.1s<1:01WARNING ⚠️ NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.4it/s 20.5s
                   all        452      73859      0.958      0.877      0.898       0.69
Speed: 2.2ms preprocess, 3.2ms inference, 0.0ms loss, 11.8ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0

In [ ]:
metrics_test = model.val(split="test")
print(metrics_test)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3441.7±853.2 MB/s, size: 947.9 KB)
val: Scanning /content/pio_dataset/labels/test... 0 images, 73 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 73/73 2.9Kit/s 0.0s
WARNING ⚠️ val: No labels found in /content/pio_dataset/labels/test.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
val: New cache created: /content/pio_dataset/labels/test.cache
WARNING ⚠️ Labels are missing or empty in /content/pio_dataset/labels/test.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.2s/it 10.8s


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:695: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:741: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:741: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all         73          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 8.3ms preprocess, 26.0ms inference, 0.0ms loss, 19.8ms postprocess per image
Results saved to /content/runs/detect/val-2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([], dtype=int64)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ef5f7634140>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.02502

In [ ]:
model_path = "/content/drive/MyDrive/PoultryVision/chicken_v1/weights/best.pt"

In [ ]:
model = YOLO(model_path)

In [ ]:
all_results_data = []

In [ ]:
onnx_filename = model.export(
    format="onnx",
    opset=18,
    imgsz=1280,
    simplify=True,
    dynamic=False
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/PoultryVision/chicken_PIO_V3/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 5, 33600) (18.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 328ms
Prepared 4 packages in 1.22s
Installed 4 packages in 241ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0